In [1]:
import os
import sys
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)
import torch
import librosa
import numpy as np
import pandas as pd

from transformers import (
    HubertModel,
    AutoProcessor
)
from tqdm import tqdm

In [3]:
from transformers import HubertModel, Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/hubert-base-ls960"
)

model = HubertModel.from_pretrained(
    "facebook/hubert-base-ls960"
)

model.eval()

print("HuBERT Loaded Successfully")

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/hubert-base-ls960 were not used when initializing HubertModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing HubertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing HubertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of HubertModel were not initialized from the model checkpoint at facebook/hubert-base-ls960 and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for pre

HuBERT Loaded Successfully


In [4]:
import os

sample_file = None

for root, dirs, files in os.walk("../Dataset"):
    for file in files:
        if file.endswith(".wav"):
            sample_file = os.path.join(root, file)
            break
    if sample_file:
        break

print(sample_file)

../Dataset\Actor_01\03-01-01-01-01-01-01.wav


In [5]:
audio, sr = librosa.load(
    sample_file,
    sr=16000
)

print(audio.shape)
print(sr)

(52853,)
16000


In [6]:
inputs = feature_extractor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

print(inputs.input_values.shape)

torch.Size([1, 52853])


In [7]:
with torch.no_grad():

    outputs = model(
        inputs.input_values
    )

embedding = outputs.last_hidden_state

print(embedding.shape)

torch.Size([1, 164, 768])


In [8]:
embedding = embedding.mean(dim=1)

print(embedding.shape)

torch.Size([1, 768])


In [9]:
embedding = embedding.squeeze().numpy()

print(embedding.shape)

(768,)


In [10]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

In [11]:
audio_files = []

for root, dirs, files in os.walk("../Dataset"):

    for file in files:

        if file.endswith(".wav"):

            audio_files.append(
                os.path.join(root, file)
            )

print("Total Audio Files:", len(audio_files))

Total Audio Files: 2880


In [12]:
def extract_hubert_embedding(file_path):

    audio, sr = librosa.load(
        file_path,
        sr=16000
    )

    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():

        outputs = model(
            inputs.input_values
        )

    embedding = outputs.last_hidden_state

    embedding = embedding.mean(
        dim=1
    )

    embedding = embedding.squeeze().numpy()

    return embedding

In [13]:
sample = extract_hubert_embedding(
    audio_files[0]
)

print(sample.shape)

(768,)


In [14]:
dataset = []

for file in tqdm(audio_files):

    embedding = extract_hubert_embedding(file)

    emotion = emotion_map[
        os.path.basename(file).split("-")[2]
    ]

    row = {}

    row["file"] = os.path.basename(file)

    for i in range(768):

        row[f"hubert_{i}"] = embedding[i]

    row["emotion"] = emotion

    dataset.append(row)

100%|██████████| 2880/2880 [12:32<00:00,  3.83it/s]


In [15]:
hubert_df = pd.DataFrame(dataset)
hubert_df.head()

,file,hubert_0,hubert_1,hubert_2,hubert_3,hubert_4,hubert_5,hubert_6,hubert_7,hubert_8,...,hubert_759,hubert_760,hubert_761,hubert_762,hubert_763,hubert_764,hubert_765,hubert_766,hubert_767,emotion
0,03-01-01-01-01-01-01.wav,0.042271,0.063757,-0.018579,0.059280,-0.284180,0.019671,0.026444,0.940023,0.055462,...,-0.217184,-0.117624,0.013609,0.015769,-0.110795,-0.080364,-0.063696,-0.109707,0.160945,neutral
1,03-01-01-01-01-02-01.wav,0.014620,0.143589,-0.007302,0.054433,-0.301745,-0.002126,0.042981,0.905375,0.023140,...,-0.314784,-0.143793,0.030687,0.003763,-0.111240,-0.096020,-0.089132,-0.115767,0.136120,neutral
2,03-01-01-01-02-01-01.wav,0.004005,0.157065,-0.006174,0.120197,-0.315657,-0.053548,0.071680,0.915667,0.049882,...,-0.352238,-0.155589,0.149152,-0.022007,-0.087848,-0.095282,-0.083677,-0.130459,0.129055,neutral
3,03-01-01-01-02-02-01.wav,0.044697,0.191159,0.002900,0.080608,-0.187402,-0.062683,0.075697,0.859285,0.052595,...,-0.343615,-0.088688,0.114904,-0.013320,-0.108208,-0.126759,-0.064620,-0.075918,0.177865,neutral
4,03-01-02-01-01-01-01.wav,0.031127,0.099100,-0.008249,0.087129,-0.283435,0.117355,0.019543,0.881802,0.002635,...,-0.306519,-0.116422,0.007911,-0.016364,-0.107425,-0.069992,-0.102666,-0.085556,0.008237,calm


In [16]:
print(hubert_df.shape)

(2880, 770)


In [17]:
output_path = r"F:\PhD\PhD_Project\Outputs\hubert_embeddings.csv"

hubert_df.to_csv(
    output_path,
    index=False
)

print("Saved Successfully")

Saved Successfully


In [18]:
saved_df = pd.read_csv(output_path)

print(saved_df.shape)

saved_df.head()

(2880, 770)


,file,hubert_0,hubert_1,hubert_2,hubert_3,hubert_4,hubert_5,hubert_6,hubert_7,hubert_8,...,hubert_759,hubert_760,hubert_761,hubert_762,hubert_763,hubert_764,hubert_765,hubert_766,hubert_767,emotion
0,03-01-01-01-01-01-01.wav,0.042271,0.063757,-0.018579,0.059280,-0.284180,0.019671,0.026444,0.940023,0.055462,...,-0.217184,-0.117624,0.013609,0.015769,-0.110795,-0.080364,-0.063696,-0.109707,0.160945,neutral
1,03-01-01-01-01-02-01.wav,0.014620,0.143589,-0.007302,0.054433,-0.301745,-0.002126,0.042981,0.905375,0.023140,...,-0.314783,-0.143793,0.030687,0.003763,-0.111240,-0.096020,-0.089132,-0.115767,0.136120,neutral
2,03-01-01-01-02-01-01.wav,0.004005,0.157065,-0.006174,0.120197,-0.315657,-0.053548,0.071680,0.915667,0.049882,...,-0.352238,-0.155589,0.149152,-0.022007,-0.087848,-0.095282,-0.083677,-0.130459,0.129055,neutral
3,03-01-01-01-02-02-01.wav,0.044697,0.191159,0.002900,0.080608,-0.187402,-0.062683,0.075697,0.859285,0.052595,...,-0.343615,-0.088688,0.114904,-0.013320,-0.108208,-0.126759,-0.064620,-0.075918,0.177865,neutral
4,03-01-02-01-01-01-01.wav,0.031127,0.099100,-0.008249,0.087129,-0.283435,0.117355,0.019543,0.881802,0.002635,...,-0.306519,-0.116422,0.007911,-0.016364,-0.107425,-0.069992,-0.102666,-0.085556,0.008237,calm
